# Week 2 — MSCOCO Image-Text Matching (N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**Dataset**: MSCOCO image-text matching (binary)
Given an (image, caption) pair, classify whether the caption MATCHES the image (1)
or is a mismatched caption from a different image (0).

**Role in the paper:** the EASY / well-aligned end of the multimodal difficulty
spectrum. CLIP was trained on image-text matching, so frozen CLIP embeddings are
near-ideal for this task. SNLI-VE is the HARD end (entailment reasoning CLIP was
not trained for). Together they bracket the range, letting you claim the
quantum-vs-classical pattern holds from well-aligned to reasoning-heavy tasks.

**Embedding:** joint 1024-dim = [CLIP image 512 ; CLIP text 512] — same as SNLI-VE
**Metric:** binary F1 | **Runtime:** ~12 min

In [ ]:
!pip install datasets transformers torch Pillow scikit-learn numpy --quiet
import numpy as np, time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import torch
from transformers import CLIPModel, CLIPProcessor
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Storage + helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
TAG = 'mscoco'
print(f"Save dir: {SAVE_DIR} | Tag: {TAG}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn): save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname
print("Helpers ready.")

---
## Step 1: Load MSCOCO + build matched/mismatched pairs

Uses `nlphuji/mscoco_2014_5k_test_image_text_retrieval` — a flat 5K set with
images and captions bundled (no fragile loader script).

**Pair construction:** for each image we make one POSITIVE pair (image + its own
caption, label 1) and one NEGATIVE pair (image + a different image's caption,
label 0). This yields a balanced 50/50 binary task.

In [ ]:
from datasets import load_dataset

print("Loading MSCOCO 5k retrieval set...")
ds = load_dataset("nlphuji/mscoco_2014_5k_test_image_text_retrieval", split='test')
print(f"Columns: {ds.column_names}")
print(f"First sample keys: {list(ds[0].keys())}")

# detect image + caption columns
cols = ds.column_names
img_col = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower()][0]
# captions often stored as a list under 'caption' or 'captions'
cap_col = 'caption' if 'caption' in cols else ('captions' if 'captions' in cols else
          [c for c in cols if 'caption' in c.lower()][0])
print(f"\nUsing image col='{img_col}', caption col='{cap_col}'")

# pull N_IMAGES distinct images + one caption each
N_IMAGES_TRAIN = 600   # -> 1200 pairs -> subsample 1000
N_IMAGES_TEST  = 130   # -> 260 pairs  -> subsample 200

def first_caption(c):
    # caption field may be a string or a list of strings
    if isinstance(c, list): return c[0] if len(c) else ""
    return c

all_imgs = []
all_caps = []
for i, s in enumerate(ds):
    all_imgs.append(s[img_col])
    all_caps.append(first_caption(s[cap_col]))
    if len(all_imgs) >= (N_IMAGES_TRAIN + N_IMAGES_TEST + 50): break

print(f"Collected {len(all_imgs)} images with captions")
print(f"Example caption: {all_caps[0][:80]}")

In [ ]:
# Build balanced matched / mismatched pairs
def build_pairs(img_indices, rng):
    pairs = []  # (img_idx, cap_idx, label)
    idx_list = list(img_indices)
    for i in idx_list:
        pairs.append((i, i, 1))                                  # matched
        j = rng.choice([k for k in idx_list if k != i])          # mismatched caption
        pairs.append((i, int(j), 0))
    rng.shuffle(pairs)
    return pairs

rng = np.random.RandomState(42)
train_img_idx = list(range(N_IMAGES_TRAIN))
test_img_idx  = list(range(N_IMAGES_TRAIN, N_IMAGES_TRAIN + N_IMAGES_TEST))

train_pairs = build_pairs(train_img_idx, rng)
test_pairs  = build_pairs(test_img_idx, rng)

print(f"Train pairs: {len(train_pairs)} | Test pairs: {len(test_pairs)}")
print(f"Train balance: {np.mean([p[2] for p in train_pairs]):.1%} matched")
LABELS = {0: 'mismatched', 1: 'matched'}

---
## Step 2: Extract CLIP joint embeddings per pair — checkpoint after

For each pair we encode the image and the (possibly mismatched) caption, then
concatenate to 1024-dim. We cache per-image and per-caption embeddings first to
avoid re-encoding the same image many times.

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_model.eval()
print("CLIP loaded.")

def encode_images(images, batch_size=32):
    embs = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        pv = clip_processor(images=batch, return_tensors='pt')['pixel_values'].to(DEVICE)
        with torch.no_grad():
            vis = clip_model.vision_model(pixel_values=pv).pooler_output
            e = clip_model.visual_projection(vis)
            e = e / torch.linalg.norm(e, dim=-1, keepdim=True)
        embs.append(e.cpu().numpy())
    return np.vstack(embs)

def encode_texts(texts, batch_size=64):
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tok = clip_processor.tokenizer(batch, return_tensors='pt', padding=True,
                                       truncation=True, max_length=77).to(DEVICE)
        with torch.no_grad():
            e = clip_model.get_text_features(**tok)
            e = e / torch.linalg.norm(e, dim=-1, keepdim=True)
        embs.append(e.cpu().numpy())
    return np.vstack(embs)

print("Encoding all images...")
t0 = time.time()
img_emb_cache = encode_images(all_imgs)   # (n_img, 512)
print(f"  images {img_emb_cache.shape} in {time.time()-t0:.0f}s")
print("Encoding all captions...")
txt_emb_cache = encode_texts(all_caps)     # (n_img, 512)
print(f"  captions {txt_emb_cache.shape}")

In [ ]:
# assemble joint embeddings per pair: [image_emb(img_idx) ; text_emb(cap_idx)]
def assemble(pairs):
    X, y = [], []
    for img_idx, cap_idx, lbl in pairs:
        joint = np.concatenate([img_emb_cache[img_idx], txt_emb_cache[cap_idx]])
        X.append(joint); y.append(lbl)
    return np.array(X), np.array(y)

train_emb, train_labels = assemble(train_pairs)
test_emb,  test_labels  = assemble(test_pairs)
print(f"Train joint emb: {train_emb.shape}  (1024-dim)")
print(f"Test  joint emb: {test_emb.shape}")
assert train_emb.shape[1] == 1024

print("\nCheckpoint 1 — raw joint embeddings:")
checkpoint('mscoco_raw.npz',
           lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                              train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA (1024 → 8) + normalize — checkpoint after

In [ ]:
N_QUBITS = 8
pca_dims = [4, 8, 12, 16]
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in pca_dims}
print("PCA variance explained (1024-dim joint -> reduced):")
for n in pca_dims:
    print(f"  {n:>2} dims -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")
pca_8 = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb); X_test_pca = pca_8.transform(test_emb)
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled  = np.clip(scaler.transform(X_test_pca), 0, np.pi)
print(f"\nTrain range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")
print("\nCheckpoint 2 — PCA + scaler:")
checkpoint('mscoco_pca_scaler.pkl',
           lambda p: pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS,
                                  'embedding_type':'clip_joint_1024','n_classes':2}, open(p,'wb')))

---
## Step 4: Classical baselines (binary F1)

In [ ]:
N_TRAIN_SVM, N_TEST_SVM = 1000, 200
assert len(X_train_scaled) >= N_TRAIN_SVM
assert len(X_test_scaled)  >= N_TEST_SVM
idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_test_scaled),  N_TEST_SVM,  replace=False)
X_tr, y_tr = X_train_scaled[idx_tr], train_labels[idx_tr]
X_te, y_te = X_test_scaled[idx_te],  test_labels[idx_te]
def bf1(yt, yp): return float(f1_score(yt, yp, average='binary'))
print(f"Train {X_tr.shape} | Test {X_te.shape} | metric=binary F1")
print(f"Class balance: {np.mean(y_tr):.1%} matched")
print()
classifiers = {
    'Linear':     SVC(kernel='linear', C=1.0, random_state=42),
    'RBF':        SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'Polynomial': SVC(kernel='poly',   C=1.0, degree=3, coef0=1, random_state=42),
}
results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'F1':>10}")
print("-"*36)
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr); yp = clf.predict(X_te)
    results[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': bf1(y_te, yp)}
    print(f"{name:<14} {results[name]['accuracy']:>10.4f} {results[name]['f1']:>10.4f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
names = list(results.keys())
for ax, key, lbl in [(axes[0],'accuracy','Accuracy'), (axes[1],'f1','Binary F1')]:
    vals = [results[n][key] for n in names]
    ax.bar(names, vals, color=['steelblue','coral','mediumseagreen'], edgecolor='white')
    ax.set_ylim(0.4, 1.0); ax.set_title(lbl); ax.set_ylabel(lbl)
    for i, v in enumerate(vals): ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
plt.suptitle('MSCOCO Matching — classical baselines (joint CLIP, N=1000)')
plt.tight_layout(); plt.show()
save_figure(fig, 'mscoco_baselines')
print("\nCheckpoint 3 — baselines:")
checkpoint('mscoco_baseline_results.pkl', lambda p: pickle.dump(results, open(p,'wb')))

---
## Step 5: Save final embeddings — shape verified

In [ ]:
def save_final(p):
    np.savez(p, X_train=X_train_scaled, X_test=X_test_scaled,
             y_train=train_labels, y_test=test_labels,
             X_tr_subset=X_tr, X_te_subset=X_te, y_tr_subset=y_tr, y_te_subset=y_te)
print("Checkpoint 4 — final embeddings:")
checkpoint('mscoco_embeddings.npz', save_final)
loaded = np.load('mscoco_embeddings.npz')
print(f"\n  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8)
assert loaded['X_te_subset'].shape == (200, 8)
print("  VERIFIED.")
meta = {'dataset':'mscoco','task':'image_text_matching','embedding_type':'clip_joint_1024',
        'n_classes':2,'metric':'binary','n_train':1000,'n_test':200,'baselines':results}
checkpoint('mscoco_meta.json', lambda p: json.dump(meta, open(p,'w'), indent=2))

---
## Week 2 done — MSCOCO Matching

### Saved:
- `mscoco_raw.npz`, `mscoco_pca_scaler.pkl`, `mscoco_baseline_results.pkl`
- `mscoco_embeddings.npz` (for Week 3), `mscoco_meta.json`, `mscoco_baselines.png/.pdf`

### Expect HIGH baselines:
CLIP is trained on image-text matching, so classical baselines may be very high
(possibly 90%+). That is the point — this is the well-aligned/easy end of the
spectrum. The question for Week 3 is whether the quantum-vs-classical gap behaves
the same here as on the harder SNLI-VE task.